# PANDA training — KAGGLE (build tile cache)



In [ ]:
DEBUG = False       # True = 100 slides / 1 epoch smoke test
DO_CACHE = True    # build the montage cache first (set False if reusing an attached cache dataset)

In [ ]:
!pip install -q timm imagecodecs

In [ ]:
import os, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    import openslide; HAVE_OPENSLIDE = True
except Exception:
    HAVE_OPENSLIDE = False
print('device:', device, '| timm', timm.__version__, '| openslide', HAVE_OPENSLIDE)

## Config


In [ ]:
data_dir   = '/kaggle/input/competitions/prostate-cancer-grade-assessment'
RAW_FOLDER = os.path.join(data_dir, 'train_images')
CACHE_DIR  = '/kaggle/working/tiles_cache'        # cache is written here, then save as a Dataset
OUT_DIR    = '/kaggle/working'

MODEL_NAME = 'effnetb0_36x256_fold0'              # model #1
BACKBONE   = 'efficientnet_b0'

fold = 0; LEVEL = 1; tile_size = 256; n_tiles = 36
batch_size = 4; num_workers = 4
init_lr = 3e-4; warmup_factor = 10; warmup_epo = 1
n_epochs = 1 if DEBUG else 25
USE_AMP = True

df_full = pd.read_csv(os.path.join(data_dir, 'train.csv'))
df_train = df_full.sample(100, random_state=42).reset_index(drop=True) if DEBUG else df_full.copy()
print('slides to train on:', len(df_train))

In [ ]:
# slide reader
def read_slide(path, level=LEVEL):
    if HAVE_OPENSLIDE:
        s = openslide.OpenSlide(path); lv = min(level, s.level_count - 1)
        img = s.read_region((0, 0), lv, s.level_dimensions[lv]).convert('RGB'); s.close()
        return np.asarray(img)
    import tifffile
    with tifffile.TiffFile(path) as tif:
        ser = tif.series[0]
        arr = ser.levels[min(level, len(ser.levels)-1)].asarray() if len(ser.levels) > 1 else ser.asarray()
    arr = np.asarray(arr)
    if arr.ndim == 2: arr = np.stack([arr]*3, -1)
    return arr[..., :3]

def get_tiles(img, mode=0):
    h, w, _ = img.shape
    pad_h = (tile_size - h % tile_size) % tile_size + ((tile_size*mode)//2)
    pad_w = (tile_size - w % tile_size) % tile_size + ((tile_size*mode)//2)
    img2 = np.pad(img, [[pad_h//2, pad_h-pad_h//2],[pad_w//2, pad_w-pad_w//2],[0,0]], constant_values=255)
    img3 = img2.reshape(img2.shape[0]//tile_size, tile_size, img2.shape[1]//tile_size, tile_size, 3)
    img3 = img3.transpose(0,2,1,3,4).reshape(-1, tile_size, tile_size, 3)
    if len(img3) < n_tiles:
        img3 = np.pad(img3, [[0, n_tiles-len(img3)],[0,0],[0,0],[0,0]], constant_values=255)
    idxs = np.argsort(img3.reshape(img3.shape[0], -1).sum(-1))[:n_tiles]
    return img3[idxs]

## Build the tile cache (one-time)


In [ ]:
from PIL import Image as PILImage
if DO_CACHE:
    os.makedirs(CACHE_DIR, exist_ok=True)
    n_row = int(np.sqrt(n_tiles)); failed = []
    for image_id in tqdm(df_train.image_id.tolist(), desc='caching'):
        out = os.path.join(CACHE_DIR, f'{image_id}.jpg')
        if os.path.exists(out): continue
        try:
            tiles = get_tiles(read_slide(os.path.join(RAW_FOLDER, f'{image_id}.tiff')), 0)
            montage = np.zeros((tile_size*n_row, tile_size*n_row, 3), np.uint8)
            for hh in range(n_row):
                for ww in range(n_row):
                    montage[hh*tile_size:(hh+1)*tile_size, ww*tile_size:(ww+1)*tile_size] = tiles[hh*n_row+ww]
            PILImage.fromarray(montage).save(out, quality=95)
        except Exception as e:
            failed.append(image_id); print('FAIL', image_id, e)
    df_full.to_csv(os.path.join(CACHE_DIR, 'train.csv'), index=False)   # ship labels with the cache
    print('cached:', len(os.listdir(CACHE_DIR)), '| failed:', len(failed))
else:
    print('reusing existing cache at', CACHE_DIR)

## Dataset (reads cached montages)

In [ ]:
# ---------- dataset that reads CACHED montages (fast: 1 JPEG per slide) ----------
from PIL import Image
import albumentations

transforms_train = albumentations.Compose([
    albumentations.Transpose(p=0.5),
    albumentations.VerticalFlip(p=0.5),
    albumentations.HorizontalFlip(p=0.5)])
transforms_val = albumentations.Compose([])

class CachedDataset(Dataset):
    def __init__(self, df, transform=None, has_label=True):
        self.df = df.reset_index(drop=True); self.transform = transform; self.has_label = has_label
    def __len__(self):
        return self.df.shape[0]
    def __getitem__(self, i):
        row = self.df.iloc[i]
        montage = np.array(Image.open(os.path.join(CACHE_DIR, f'{row.image_id}.jpg')))
        montage = 255 - montage                       # invert (background -> 0)
        if self.transform is not None:
            montage = self.transform(image=montage)['image']
        montage = (montage.astype(np.float32) / 255).transpose(2, 0, 1)
        if self.has_label:
            label = np.zeros(5, np.float32); label[:int(row.isup_grade)] = 1.
            return torch.tensor(montage), torch.tensor(label)
        return torch.tensor(montage)

## Stratified folds

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
df_train['fold'] = -1
for i, (_, va) in enumerate(skf.split(df_train, df_train.isup_grade)):
    df_train.loc[va, 'fold'] = i
df_this  = df_train[df_train.fold != fold]
df_valid = df_train[df_train.fold == fold].reset_index(drop=True)
print('train', len(df_this), '| val', len(df_valid))

## Model, loss, metric

In [ ]:
class enetv2(nn.Module):
    def __init__(self, backbone=BACKBONE, out_dim=5, pretrained=True):
        super().__init__()
        self.enet = timm.create_model(backbone, pretrained=pretrained,
                                      num_classes=0, global_pool='avg')
        self.myfc = nn.Linear(self.enet.num_features, out_dim)
    def forward(self, x):
        return self.myfc(self.enet(x))

criterion = nn.BCEWithLogitsLoss()
def safe_qwk(pred, true):
    return cohen_kappa_score(pred, true, weights='quadratic') if len(pred) else float('nan')

## Train / val loops (with batch progress bars)

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_epoch(loader, optimizer, epoch):
    model.train(); losses = []
    pbar = tqdm(loader, desc=f'epoch {epoch}/{n_epochs}')
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss = criterion(model(data), target)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        losses.append(loss.item()); pbar.set_postfix(loss=float(np.mean(losses[-20:])))
    return float(np.mean(losses))

def val_epoch(loader, df_valid, get_output=False):
    model.eval(); losses, P, T = [], [], []
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            logits = model(data); losses.append(criterion(logits, target).item())
            P.append(logits.sigmoid().sum(1).round()); T.append(target.sum(1))
    P = torch.cat(P).cpu().numpy(); T = torch.cat(T).cpu().numpy()
    if get_output:
        return P, T
    mk = (df_valid.data_provider == 'karolinska').values
    mr = (df_valid.data_provider == 'radboud').values
    return (float(np.mean(losses)), (P == T).mean()*100, safe_qwk(P, T),
            safe_qwk(P[mk], df_valid.isup_grade.values[mk]),
            safe_qwk(P[mr], df_valid.isup_grade.values[mr]))

In [ ]:
from IPython.display import clear_output

def live_plot(history):
    clear_output(wait=True)
    ep = range(1, len(history['train_loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(ep, history['train_loss'], '-o', label='train')
    ax[0].plot(ep, history['val_loss'], '-o', label='val')
    ax[0].set_title('BCE loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
    ax[1].plot(ep, history['qwk'],   '-o', label='QWK all')
    ax[1].plot(ep, history['qwk_k'], '--', label='Karolinska')
    ax[1].plot(ep, history['qwk_r'], '--', label='Radboud')
    ax[1].set_ylim(0, 1); ax[1].set_title('Quadratic weighted kappa')
    ax[1].set_xlabel('epoch'); ax[1].legend()
    plt.tight_layout(); plt.show()

## Train — live curves update every epoch

In [ ]:
train_loader = DataLoader(CachedDataset(df_this,  transforms_train),
                          batch_size=batch_size, shuffle=True,  num_workers=num_workers)
valid_loader = DataLoader(CachedDataset(df_valid, transforms_val),
                          batch_size=batch_size, shuffle=False, num_workers=num_workers)

model = enetv2(pretrained=True).to(device)
optimizer = optim.Adam(model.parameters(), lr=init_lr)
warmup = LinearLR(optimizer, start_factor=1/warmup_factor, total_iters=warmup_epo)
cosine = CosineAnnealingLR(optimizer, T_max=max(1, n_epochs - warmup_epo))
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_epo])

history = {'train_loss': [], 'val_loss': [], 'qwk': [], 'qwk_k': [], 'qwk_r': []}
qwk_max = -1.; best_file = os.path.join(OUT_DIR, f'{MODEL_NAME}_best.pth')
for epoch in range(1, n_epochs + 1):
    tl = train_epoch(train_loader, optimizer, epoch)
    vl, acc, qwk, qwk_k, qwk_r = val_epoch(valid_loader, df_valid)
    scheduler.step()
    for k, v in zip(history, [tl, vl, qwk, qwk_k, qwk_r]): history[k].append(v)
    live_plot(history)                                   # <-- live display each epoch
    print(f'epoch {epoch:2d} | train {tl:.4f} | val {vl:.4f} | acc {acc:.1f} | '
          f'QWK {qwk:.4f} (K {qwk_k:.3f} / R {qwk_r:.3f})')
    if qwk > qwk_max:
        qwk_max = qwk; torch.save(model.state_dict(), best_file); print('   saved best ->', best_file)
print('BEST validation QWK:', round(qwk_max, 4))

## Validation confusion matrix

In [ ]:
model.load_state_dict(torch.load(best_file)); model.to(device)
P, T = val_epoch(valid_loader, df_valid, get_output=True)
cm = confusion_matrix(T, P, labels=list(range(6)))
fig, ax = plt.subplots(figsize=(5.5, 4.5)); im = ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, int(v), ha='center', va='center', color='white' if v > cm.max()/2 else 'black')
ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'Validation confusion (QWK={safe_qwk(P, T):.3f})'); fig.colorbar(im)
plt.tight_layout(); plt.show()